# NanoMamba: Noise-Robust KWS by Architectural Design
**IEEE/ACM TASLP — Complete Training & Evaluation Pipeline**

---

## Execution Order

| Step | Cell | Description | Time |
|------|------|-------------|------|
| **0** | Setup | GPU check + Mount Drive | 30s |
| **QR** | **Quick Recovery** | **Clone+Data+Ckpt+GTCRN one-shot (after runtime reset)** | **~3min** |
| **1** | Clone | Latest code from GitHub | 10s |
| **2** | Dataset | Google Speech Commands V2 download | 2min |
| **3** | Verify | All model forward pass check | 10s |
| **4** | Train Baselines | DS-CNN-S (23.7K) + BC-ResNet-1 (7.5K) | ~10min |
| **5** | Train NanoMamba | Tiny (4.6K) + Small (12K) | ~10min |
| **6** | Train DualPCEN | NM-Tiny-DualPCEN (4.9K) | ~8min |
| **6b** | Train Matched+TriPCEN | Matched-Dual (7.4K) + Tiny-Tri (5.1K) + Matched-Tri (7.4K) | ~25min |
| **6c** | **Train v2 Enhanced** | **DualPCEN-v2 + TriPCEN-v2 (TMI+SNR routing)** | **~35min** |
| **6d** | **Train v2+SSMv2** | **Full v2 stack: PCEN v2 + SA-SSM v2 + curriculum v2** | **~35min** |
| **7** | Noise Eval | All models x 5 noise types x 7 SNRs | ~60min |
| **8** | GTCRN Setup | Clone GTCRN (23.7K pre-trained enhancer) | 10s |
| **9** | SS+Bypass Eval | SS v2 + Bypass v2 evaluation | ~30min |
| **10** | **v1 vs v2 Comparison** | **Same params, enhanced routing** | **~30min** |
| **10b** | **v2 vs v2+SSMv2** | **Full stack comparison + continuous calibration** | **~30min** |
| **11** | Results | Summary tables + comparison plots | 1min |
| **12** | Backup | Save checkpoints to Drive | 30s |

**Runtime reset? Just run QR cell, then jump to Step 7!**

## Models

| Model | Params | Type | Key Feature |
|-------|--------|------|-------------|
| `NanoMamba-Tiny` | 4,634 | SSM | SA-SSM baseline |
| `NanoMamba-Small` | 12,035 | SSM | SA-SSM larger |
| `NanoMamba-Tiny-DualPCEN` | 4,957 | SSM | Dual-PCEN + SF routing |
| `NanoMamba-Matched-DualPCEN` | 7,402 | SSM | Dual-PCEN param-matched |
| `NanoMamba-Tiny-TriPCEN` | 5,118 | SSM | 3-Expert PCEN |
| `NanoMamba-Matched-TriPCEN` | 7,414 | SSM | 3-Expert param-matched |
| **`NanoMamba-Tiny-DualPCEN-v2`** | **4,957** | **SSM** | **v2: TMI + SNR-Cond routing** |
| **`NanoMamba-Matched-DualPCEN-v2`** | **7,402** | **SSM** | **v2: Enhanced param-matched** |
| **`NanoMamba-Tiny-TriPCEN-v2`** | **5,118** | **SSM** | **v2: 3-Expert enhanced** |
| **`NanoMamba-Matched-TriPCEN-v2`** | **7,414** | **SSM** | **v2: 3-Expert enhanced matched** |
| **`NanoMamba-Tiny-DualPCEN-v2-SSMv2`** | **4,957** | **SSM** | **Full v2: PCEN+SSM v2** |
| **`NanoMamba-Matched-DualPCEN-v2-SSMv2`** | **7,402** | **SSM** | **Full v2: param-matched** |
| **`NanoMamba-Tiny-TriPCEN-v2-SSMv2`** | **5,118** | **SSM** | **Full v2: 3-Expert** |
| **`NanoMamba-Matched-TriPCEN-v2-SSMv2`** | **7,414** | **SSM** | **Full v2: 3-Expert matched** |
| `DS-CNN-S` | 23,700 | CNN | Depthwise Separable CNN baseline |
| `BC-ResNet-1` | 7,464 | CNN | Broadcasted Residual Net baseline |

---
## Step 0: Setup & GPU Check

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("WARNING: No GPU! Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## Quick Recovery (Runtime Reset? Run THIS cell only!)
Clone + Dataset + Checkpoints + GTCRN in one shot.
Skip if you already ran Steps 0-2.

In [ ]:
import torch, os, sys, shutil, subprocess

# ===== 0. Fix CWD (after runtime reset, old dir may be gone) =====
try:
    os.getcwd()
except OSError:
    os.chdir('/content')

# ===== 1. GPU =====
if torch.cuda.is_available():
    print(f"[1/7] GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB)")
else:
    print("[1/7] WARNING: No GPU!")

# ===== 2. Drive =====
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("[2/7] Drive mounted")

# ===== 3. Clone =====
os.chdir('/content')  # ensure valid CWD before subprocess
subprocess.run(['rm', '-rf', '/content/NanoMamba'])
result = subprocess.run(
    ['git', 'clone', 'https://github.com/DrJinHoChoi/NanoMamba-TASLP.git', '/content/NanoMamba'],
    capture_output=True, text=True, cwd='/content')
if result.returncode == 0:
    os.chdir('/content/NanoMamba')
    commit = subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip()
    print(f"[3/7] Code cloned ({commit})")
else:
    print(f"[3/7] Clone FAILED: {result.stderr}")

# ===== 4. Checkpoints from Drive =====
DRIVE_CKPT = '/content/drive/MyDrive/NanoMamba/checkpoints_full'
LOCAL_CKPT = '/content/NanoMamba/checkpoints_full'
if os.path.exists(DRIVE_CKPT):
    shutil.copytree(DRIVE_CKPT, LOCAL_CKPT, dirs_exist_ok=True)
    models = [d for d in os.listdir(LOCAL_CKPT) if os.path.isdir(os.path.join(LOCAL_CKPT, d))]
    print(f"[4/7] Checkpoints: {models}")
else:
    os.makedirs(LOCAL_CKPT, exist_ok=True)
    print("[4/7] No checkpoints on Drive (will need training)")

# ===== 5. Dataset =====
DATA_DIR = '/content/NanoMamba/data'
os.makedirs(DATA_DIR, exist_ok=True)
sys.path.insert(0, '/content/NanoMamba')
from train_colab import SpeechCommandsDataset
train_ds = SpeechCommandsDataset(DATA_DIR, subset='training', augment=True)
val_ds = SpeechCommandsDataset(DATA_DIR, subset='validation', augment=False)
test_ds = SpeechCommandsDataset(DATA_DIR, subset='testing', augment=False)
print(f"[5/7] Dataset: Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}")

# ===== 6. GTCRN =====
if not os.path.exists('/content/gtcrn'):
    subprocess.run(['git', 'clone', 'https://github.com/Xiaobin-Rong/gtcrn.git', '/content/gtcrn'],
                   capture_output=True, cwd='/content')
print(f"[6/7] GTCRN: {'Ready' if os.path.exists('/content/gtcrn/checkpoints') else 'MISSING'}")

# ===== 7. Verify strict=False =====
with open('/content/NanoMamba/train_colab.py') as f:
    n = f.read().count('strict=False')
print(f"[7/7] strict=False: {n} locations")

# ===== Summary =====
print("\n" + "=" * 55)
print("  SETUP COMPLETE — Jump to Step 4 (train) or 7 (eval)")
print("=" * 55)

## Step 1: Clone Latest Code from GitHub

In [ ]:
import os
try:
    os.getcwd()
except OSError:
    os.chdir('/content')

# Clean clone (always gets latest code)
!rm -rf /content/NanoMamba
!cd /content && git clone https://github.com/DrJinHoChoi/NanoMamba-TASLP.git /content/NanoMamba
%cd /content/NanoMamba
!git log --oneline -3
print("\n--- Files ---")
!ls *.py

## Step 2: Download Dataset (Google Speech Commands V2)

In [ ]:
import os, sys
sys.path.insert(0, '/content/NanoMamba')

DATA_DIR = '/content/NanoMamba/data'
CKPT_DIR = '/content/NanoMamba/checkpoints_full'
os.makedirs(DATA_DIR, exist_ok=True)

from train_colab import SpeechCommandsDataset

print("Loading datasets...")
train_ds = SpeechCommandsDataset(DATA_DIR, subset='training', augment=True)
val_ds = SpeechCommandsDataset(DATA_DIR, subset='validation', augment=False)
test_ds = SpeechCommandsDataset(DATA_DIR, subset='testing', augment=False)
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

## Step 3: Verify All Models (Forward Pass Check)

In [ ]:
from nanomamba import (
    create_nanomamba_tiny, create_nanomamba_small,
    create_nanomamba_tiny_pcen, create_nanomamba_tiny_dualpcen,
    create_nanomamba_small_dualpcen,
    create_nanomamba_matched_dualpcen,
    create_nanomamba_tiny_tripcen,
    create_nanomamba_matched_tripcen,
    # v2 Enhanced Routing
    create_nanomamba_tiny_dualpcen_v2,
    create_nanomamba_matched_dualpcen_v2,
    create_nanomamba_tiny_tripcen_v2,
    create_nanomamba_matched_tripcen_v2,
    # v2 + SSM v2 (full stack)
    create_nanomamba_tiny_dualpcen_v2_ssmv2,
    create_nanomamba_matched_dualpcen_v2_ssmv2,
    create_nanomamba_tiny_tripcen_v2_ssmv2,
    create_nanomamba_matched_tripcen_v2_ssmv2,
)
from train_colab import DSCNN_S, BCResNet

audio = torch.randn(2, 16000)

print("=" * 80)
print("  Model Verification (v1 + v2 + v2-SSMv2)")
print("=" * 80)

for name, m in [
    ('NanoMamba-Tiny',                create_nanomamba_tiny()),
    ('NanoMamba-Small',               create_nanomamba_small()),
    ('NM-Tiny-DualPCEN',             create_nanomamba_tiny_dualpcen()),
    ('NM-Matched-DualPCEN',          create_nanomamba_matched_dualpcen()),
    ('NM-Tiny-TriPCEN',              create_nanomamba_tiny_tripcen()),
    ('NM-Matched-TriPCEN',           create_nanomamba_matched_tripcen()),
    ('NM-Tiny-DualPCEN-v2',          create_nanomamba_tiny_dualpcen_v2()),
    ('NM-Matched-DualPCEN-v2',       create_nanomamba_matched_dualpcen_v2()),
    ('NM-Tiny-TriPCEN-v2',           create_nanomamba_tiny_tripcen_v2()),
    ('NM-Matched-TriPCEN-v2',        create_nanomamba_matched_tripcen_v2()),
    ('NM-Tiny-DualPCEN-v2-SSMv2 *',  create_nanomamba_tiny_dualpcen_v2_ssmv2()),
    ('NM-Matched-DualPCEN-v2-SSMv2 *', create_nanomamba_matched_dualpcen_v2_ssmv2()),
    ('NM-Tiny-TriPCEN-v2-SSMv2 *',   create_nanomamba_tiny_tripcen_v2_ssmv2()),
    ('NM-Matched-TriPCEN-v2-SSMv2 *', create_nanomamba_matched_tripcen_v2_ssmv2()),
    ('DS-CNN-S',                     DSCNN_S()),
    ('BC-ResNet-1',                  BCResNet(scale=1)),
]:
    m.eval()
    p = sum(x.numel() for x in m.parameters())
    try:
        with torch.no_grad():
            out = m(audio)
        status = f"output={list(out.shape)}"
    except:
        status = "(mel input - OK)"
    print(f"  {name:<40} | {p:>6,} params ({p*4/1024:.1f}KB) | {status}")

print("\n  All models verified!")

## Step 4: Train CNN Baselines (DS-CNN-S, BC-ResNet-1)

In [ ]:
# DS-CNN-S: 23.7K params (~5min)
!python train_colab.py \
    --models DS-CNN-S \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --batch_size 128 \
    --lr 3e-3 \
    --noise_aug --calibrate

In [ ]:
# BC-ResNet-1: 7.5K params (~5min)
!python train_colab.py \
    --models BC-ResNet-1 \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --batch_size 128 \
    --lr 3e-3 \
    --noise_aug --calibrate

## Step 5: Train NanoMamba Baselines (Tiny, Small)

In [ ]:
# NanoMamba-Tiny: 4,634 params (~5min)
import os
if os.path.exists('./checkpoints_full/NanoMamba-Tiny/best.pt'):
    print("NanoMamba-Tiny checkpoint already exists, skipping training.")
    print("Delete checkpoints_full/NanoMamba-Tiny/ to retrain.")
else:
    !python train_colab.py \
        --models NanoMamba-Tiny \
        --data_dir ./data \
        --checkpoint_dir ./checkpoints_full \
        --epochs 30 \
        --noise_aug --calibrate

In [ ]:
# NanoMamba-Small: 12,035 params (~8min)
import os
if os.path.exists('./checkpoints_full/NanoMamba-Small/best.pt'):
    print("NanoMamba-Small checkpoint already exists, skipping training.")
else:
    !python train_colab.py \
        --models NanoMamba-Small \
        --data_dir ./data \
        --checkpoint_dir ./checkpoints_full \
        --epochs 30 \
        --lr 1e-3 \
        --noise_aug --calibrate

## Step 6: Train NanoMamba-Tiny-DualPCEN (Proposed Model)

In [ ]:
# NanoMamba-Tiny-DualPCEN: 4,957 params — Dual-PCEN + SF routing
!python train_colab.py \
    --models NanoMamba-Tiny-DualPCEN \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --batch_size 128 \
    --lr 3e-3 \
    --noise_aug --calibrate

## Step 6b: Train Matched + TriPCEN Models (NEW)
Parameter-matched to BC-ResNet-1 (7,464 params) for fair comparison.
- **Matched-DualPCEN**: 7,402 params (d=21, s=5) — 2-expert PCEN
- **Tiny-TriPCEN**: ~5,118 params — 3-expert PCEN (factory/street specialist)
- **Matched-TriPCEN**: 7,414 params (d=20, s=6) — 3-expert param-matched

In [ ]:
# Matched-DualPCEN: 7,402 params (~8min)
!python train_colab.py \
    --models NanoMamba-Matched-DualPCEN \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --calibrate

# Tiny-TriPCEN: ~5,118 params — 3-expert PCEN (~8min)
!python train_colab.py \
    --models NanoMamba-Tiny-TriPCEN \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --calibrate

# Matched-TriPCEN: 7,414 params — 3-expert param-matched (~8min)
!python train_colab.py \
    --models NanoMamba-Matched-TriPCEN \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --calibrate

In [ ]:
# === v2 DualPCEN variants (same params as v1, enhanced routing) ===

# Tiny-DualPCEN-v2: 4,957 params (~8min)
!python train_colab.py \
    --models NanoMamba-Tiny-DualPCEN-v2 \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --calibrate

# Matched-DualPCEN-v2: 7,402 params (~8min)
!python train_colab.py \
    --models NanoMamba-Matched-DualPCEN-v2 \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --calibrate

# === v2 TriPCEN variants ===

# Tiny-TriPCEN-v2: 5,118 params (~8min)
!python train_colab.py \
    --models NanoMamba-Tiny-TriPCEN-v2 \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --calibrate

# Matched-TriPCEN-v2: 7,414 params (~8min)
!python train_colab.py \
    --models NanoMamba-Matched-TriPCEN-v2 \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --calibrate

## Step 6d: Train v2+SSMv2 (Full Stack: PCEN v2 + SA-SSM v2 + Noise Curriculum v2)
Full v2 stack: PCEN v2 (TMI+SNR routing+aux loss) + SA-SSM v2 (SNR re-normalization+PCEN gate conditioning) + Noise curriculum v2 (type-staged+SNR annealing).
**Same parameter count as v1 models!**

| Model | Params | Improvements |
|-------|--------|-------------|
| Tiny-DualPCEN-v2-SSMv2 | 4,957 | PCEN v2 routing + SA-SSM v2 + curriculum v2 |
| Matched-DualPCEN-v2-SSMv2 | 7,402 | Same, param-matched to BC-ResNet |
| Tiny-TriPCEN-v2-SSMv2 | 5,118 | 3-expert v2 + SA-SSM v2 |
| Matched-TriPCEN-v2-SSMv2 | 7,414 | 3-expert matched + SA-SSM v2 |

In [ ]:
# === v2+SSMv2: Full stack (PCEN v2 + SA-SSM v2 + Noise Curriculum v2) ===

# Tiny-DualPCEN-v2-SSMv2: 4,957 params (~8min)
!python train_colab.py \n    --models NanoMamba-Tiny-DualPCEN-v2-SSMv2 \n    --data_dir ./data \n    --checkpoint_dir ./checkpoints_full \n    --epochs 30 \n    --noise_aug --calibrate \n    --noise_curriculum_v2

# Matched-DualPCEN-v2-SSMv2: 7,402 params (~8min)
!python train_colab.py \n    --models NanoMamba-Matched-DualPCEN-v2-SSMv2 \n    --data_dir ./data \n    --checkpoint_dir ./checkpoints_full \n    --epochs 30 \n    --noise_aug --calibrate \n    --noise_curriculum_v2

# Tiny-TriPCEN-v2-SSMv2: 5,118 params (~8min)
!python train_colab.py \n    --models NanoMamba-Tiny-TriPCEN-v2-SSMv2 \n    --data_dir ./data \n    --checkpoint_dir ./checkpoints_full \n    --epochs 30 \n    --noise_aug --calibrate \n    --noise_curriculum_v2

# Matched-TriPCEN-v2-SSMv2: 7,414 params (~8min)
!python train_colab.py \n    --models NanoMamba-Matched-TriPCEN-v2-SSMv2 \n    --data_dir ./data \n    --checkpoint_dir ./checkpoints_full \n    --epochs 30 \n    --noise_aug --calibrate \n    --noise_curriculum_v2

# Evaluate ALL trained models on 5 noise types x 7 SNR levels (no enhancer)
!python train_colab.py \
    --models NanoMamba-Tiny,NanoMamba-Small,NanoMamba-Tiny-DualPCEN,NanoMamba-Matched-DualPCEN,NanoMamba-Tiny-TriPCEN,NanoMamba-Matched-TriPCEN,NanoMamba-Tiny-DualPCEN-v2,NanoMamba-Matched-DualPCEN-v2,NanoMamba-Tiny-TriPCEN-v2,NanoMamba-Matched-TriPCEN-v2,DS-CNN-S,BC-ResNet-1 \
    --eval_only \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 --calibrate

## Step 7: Noise Robustness Evaluation (All Models, No Enhancer)

In [ ]:
# Evaluate ALL trained models on 5 noise types x 7 SNR levels (no enhancer)
!python train_colab.py \
    --models NanoMamba-Tiny,NanoMamba-Small,NanoMamba-Tiny-DualPCEN,NanoMamba-Matched-DualPCEN,NanoMamba-Tiny-TriPCEN,NanoMamba-Matched-TriPCEN,DS-CNN-S,BC-ResNet-1 \
    --eval_only \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 --calibrate

## Step 7b: Reverb Evaluation (No Enhancer)
Reverb-only (RT60 = 0.2, 0.4, 0.6, 0.8s) + Noise+Reverb combined (factory/babble × SNR 0,5dB × RT60 0.3,0.6s)

In [ ]:
# Reverb evaluation (no enhancer):
# C. Reverb-only: RT60 = 0.2, 0.4, 0.6, 0.8s
# E. Noise+Reverb: factory/babble × SNR 0,5dB × RT60 0.3,0.6s
!python train_colab.py \
    --models NanoMamba-Tiny,NanoMamba-Small,NanoMamba-Tiny-DualPCEN,DS-CNN-S,BC-ResNet-1 \
    --eval_only \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --use_reverb --rt60 0.2,0.4,0.6,0.8 \
    --noise_types factory,babble \
    --snr_range=-15,-10,-5,0,5,10,15

## Step 8: Setup GTCRN Pre-trained Enhancer (23.7K params)

In [ ]:
# GTCRN: ultra-lightweight speech enhancement (ICASSP 2024)
# 23.7K params, trained on DNS3 dataset
!rm -rf /content/gtcrn
!git clone https://github.com/Xiaobin-Rong/gtcrn.git /content/gtcrn
!ls /content/gtcrn/checkpoints/
print("\nGTCRN ready!")

## Step 9: SS v2 + Bypass v2 Evaluation (NEW)
Adaptive Spectral Subtraction + Noise-Aware Bypass evaluation.
- **SS v2**: Running min noise est + per-frame adaptive oversubtract + freq-weighted floor
- **Bypass v2**: Spectral-Flatness-aware adaptive threshold + steeper sigmoid

In [ ]:
# v1 vs v2 head-to-head: Matched models (7,402 / 7,414 params)
!python train_colab.py \
    --models NanoMamba-Matched-DualPCEN,NanoMamba-Matched-DualPCEN-v2,NanoMamba-Matched-TriPCEN,NanoMamba-Matched-TriPCEN-v2,BC-ResNet-1 \
    --eval_only \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 --calibrate

# v1 vs v2 head-to-head: Tiny models (4,957 / 5,118 params)
!python train_colab.py \
    --models NanoMamba-Tiny-DualPCEN,NanoMamba-Tiny-DualPCEN-v2,NanoMamba-Tiny-TriPCEN,NanoMamba-Tiny-TriPCEN-v2 \
    --eval_only \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 --calibrate

## Step 10: v1 vs v2 Routing Comparison (Same Params)
Direct comparison: identical parameter count, only routing mechanism differs.
- v1: SF + Tilt routing (1-2 learnable gate temps)
- v2: SF + Tilt + **TMI + SNR-Cond Temp + Temporal Smoothing + Aux Loss**

In [ ]:
# SS v2 + Bypass v2: All models with adaptive spectral subtraction
!python train_colab.py \
    --models NanoMamba-Matched-DualPCEN,NanoMamba-Matched-TriPCEN,NanoMamba-Tiny-DualPCEN,NanoMamba-Tiny-TriPCEN,BC-ResNet-1 \
    --eval_only \
    --use_enhancer --enhancer_bypass \
    --ss_version v2 --bypass_version v2 \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 --calibrate

## Step 10b: v2 vs v2+SSMv2 Comparison (+ Continuous Calibration)
Full v2 stack comparison with continuous calibration.
- **v2**: PCEN v2 (TMI+SNR routing+aux loss) only
- **v2+SSMv2**: v2 + SA-SSM v2 (SNR re-norm+PCEN gate) + noise curriculum v2
- **Continuous cal**: Smooth SNR interpolation (no profile boundaries)


In [ ]:
# v2 vs v2+SSMv2 head-to-head: Matched models with continuous calibration
!python train_colab.py \n    --models NanoMamba-Matched-DualPCEN-v2,NanoMamba-Matched-DualPCEN-v2-SSMv2,NanoMamba-Matched-TriPCEN-v2,NanoMamba-Matched-TriPCEN-v2-SSMv2,BC-ResNet-1 \n    --eval_only \n    --data_dir ./data \n    --checkpoint_dir ./checkpoints_full \n    --noise_types factory,white,babble,street,pink \n    --snr_range=-15,-10,-5,0,5,10,15 --calibrate --calibrate_continuous

# v2 vs v2+SSMv2: Tiny models with continuous calibration
!python train_colab.py \n    --models NanoMamba-Tiny-DualPCEN-v2,NanoMamba-Tiny-DualPCEN-v2-SSMv2,NanoMamba-Tiny-TriPCEN-v2,NanoMamba-Tiny-TriPCEN-v2-SSMv2 \n    --eval_only \n    --data_dir ./data \n    --checkpoint_dir ./checkpoints_full \n    --noise_types factory,white,babble,street,pink \n    --snr_range=-15,-10,-5,0,5,10,15 --calibrate --calibrate_continuous

In [ ]:
# SS v1 + Bypass v1 (baseline for comparison with v2)
!python train_colab.py \
    --models NanoMamba-Matched-DualPCEN,NanoMamba-Matched-TriPCEN,NanoMamba-Tiny-DualPCEN,NanoMamba-Tiny-TriPCEN,BC-ResNet-1 \
    --eval_only \
    --use_enhancer --enhancer_bypass \
    --ss_version v1 --bypass_version v1 \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 --calibrate

## Step 9b: Reverb Evaluation WITH GTCRN Enhancer
D. Reverb+GTCRN (RT60 = 0.2, 0.4, 0.6, 0.8s) + F. Noise+Reverb+GTCRN combined

In [ ]:
# Reverb evaluation WITH GTCRN enhancer:
# D. Reverb+GTCRN: RT60 = 0.2, 0.4, 0.6, 0.8s
# F. Noise+Reverb+GTCRN: factory/babble × SNR 0,5dB × RT60 0.3,0.6s
!python train_colab.py \
    --models NanoMamba-Tiny,NanoMamba-Small,NanoMamba-Tiny-DualPCEN,DS-CNN-S,BC-ResNet-1 \
    --eval_only \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --use_reverb --rt60 0.2,0.4,0.6,0.8 \
    --use_enhancer --enhancer_type gtcrn --gtcrn_dir /content/gtcrn \
    --noise_types factory,babble \
    --snr_range=-15,-10,-5,0,5,10,15

## Step 10: Results Summary & Comparison Plot

In [ ]:
import json, os

results_path = './results/final_results.json'
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    
    print("=" * 80)
    print("  FINAL RESULTS SUMMARY")
    print("=" * 80)
    
    for model_name, data in results.get('models', {}).items():
        print(f"\n  {model_name}: {data.get('params', '?'):,} params")
        print(f"    Test Accuracy: {data.get('test_acc', 0):.2f}%")
        for noise_type, snr_data in data.get('noise_robustness', {}).items():
            snrs = ['-15', '-10', '-5', '0', '5', '10', '15', 'clean']
            vals = [snr_data.get(s, 0) for s in snrs]
            print(f"    {noise_type:<8}: " + " | ".join(f"{v:.1f}" for v in vals))
else:
    print("No results file found. Run evaluation cells first.")

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# Known results (from previous experiments) as fallback
known = {
    'NanoMamba-Tiny (4.6K)': {
        'factory': [38.4, 56.1, 70.1, 77.6, 83.2, 85.1, 86.6],
        'white':   [20.2, 51.6, 69.3, 79.8, 86.2, 90.1, 91.8],
        'babble':  [58.6, 60.4, 65.0, 69.6, 77.3, 84.1, 87.4],
        'color': '#F4845F', 'marker': 'D', 'ls': '--',
    },
    'DS-CNN-S (23.7K)': {
        'factory': [59.2, 62.6, 66.4, 75.6, 83.9, 90.7, 93.3],
        'white':   [11.1, 12.0, 11.3, 13.9, 30.0, 55.6, 75.3],
        'babble':  [34.9, 45.7, 55.4, 70.1, 81.0, 88.8, 92.8],
        'color': '#457B9D', 'marker': 'o', 'ls': '-.',
    },
    'BC-ResNet-1 (7.5K)': {
        'factory': [57.1, 61.5, 65.5, 71.6, 78.3, 83.8, 87.7],
        'white':   [22.0, 25.0, 37.8, 54.7, 66.1, 75.5, 84.4],
        'babble':  [37.9, 46.6, 58.0, 73.7, 85.0, 91.5, 94.1],
        'color': '#2A9D8F', 'marker': '^', 'ls': ':',
    },
}

snr = [-15, -10, -5, 0, 5, 10, 15]
noises = ['factory', 'white', 'babble']
titles = ['(a) Factory (Stationary)', '(b) White (Broadband)', '(c) Babble (Non-stationary)']

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

for idx, (noise, title) in enumerate(zip(noises, titles)):
    ax = axes[idx]
    for name, d in known.items():
        ax.plot(snr, d[noise], color=d['color'], marker=d['marker'],
                ls=d['ls'], lw=2, markersize=7, label=name)
    
    # Placeholder for DualPCEN (will be filled after training)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('SNR (dB)')
    if idx == 0: ax.set_ylabel('Accuracy (%)')
    ax.set_xticks(snr)
    ax.set_ylim(0, 100)
    ax.grid(True, alpha=0.3)
    ax.axvspan(-15, -5, alpha=0.05, color='red')
    ax.axvspan(-5, 15, alpha=0.03, color='green')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, fontsize=10,
           bbox_to_anchor=(0.5, -0.04))
plt.suptitle('Noise Robustness: NanoMamba-Tiny-DualPCEN vs Baselines',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results_comparison.png")

## Step 11: Save Checkpoints to Google Drive

In [ ]:
import shutil, os

DRIVE_DIR = '/content/drive/MyDrive/NanoMamba'
os.makedirs(DRIVE_DIR, exist_ok=True)

# Copy checkpoints
src = '/content/NanoMamba/checkpoints_full'
dst = os.path.join(DRIVE_DIR, 'checkpoints_full')
if os.path.exists(src):
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"Checkpoints saved to {dst}")

# Copy results
src_r = '/content/NanoMamba/results'
dst_r = os.path.join(DRIVE_DIR, 'results')
if os.path.exists(src_r):
    shutil.copytree(src_r, dst_r, dirs_exist_ok=True)
    print(f"Results saved to {dst_r}")

# List saved checkpoints
print("\n--- Saved Checkpoints ---")
for root, dirs, files in os.walk(dst):
    for f in files:
        fp = os.path.join(root, f)
        sz = os.path.getsize(fp) / 1024
        print(f"  {os.path.relpath(fp, dst):<40} {sz:.1f} KB")

print("\nDone! All saved to Google Drive.")

---
## (Optional) Single-PCEN Variants

In [ ]:
# NanoMamba-Tiny-PCEN (single delta=0.01, factory specialist)
# !python train_colab.py \
#     --models NanoMamba-Tiny-PCEN \
#     --data_dir ./data \
#     --checkpoint_dir ./checkpoints_full \
#     --epochs 30 \
#     --noise_types factory,white,babble \
#     --snr_range=-15,-10,-5,0,5,10,15

In [ ]:
# Spectral subtraction enhancer (0 params, classical baseline)
# !python train_colab.py \
#     --models NanoMamba-Tiny,DS-CNN-S \
#     --eval_only \
#     --use_enhancer --enhancer_type spectral \
#     --checkpoint_dir ./checkpoints_full \
#     --noise_types factory,white,babble \
#     --snr_range=-15,-10,-5,0,5,10,15